# Evaluate Recent Tropical Cyclones

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brightbandtech/extremeweatherbench/blob/main/notebooks/evaluate_recent_tropical_cyclones.ipynb)

This notebook demonstrates how to evaluate tropical cyclone track and landfall metrics using EWB. Uses a CIRA forecast with IBTrACS best-track targets. Optimized for Google Colab with a small demo case.

In [ ]:
!pip install -q extremeweatherbench

In [ ]:
import datetime
import extremeweatherbench as ewb
from extremeweatherbench import inputs, derived, metrics
from extremeweatherbench.cases import IndividualCase
from extremeweatherbench.regions import BoundingBoxRegion

# Mini-case: Hurricane Ida 2021 — Colab-optimized
demo_case = IndividualCase(
    case_id_number=9006,
    title="Hurricane Ida 2021 (demo)",
    start_date=datetime.datetime(2021, 8, 28),
    end_date=datetime.datetime(2021, 8, 31),
    location=BoundingBoxRegion.create_region(
        latitude_min=27.0,
        latitude_max=33.0,
        longitude_min=268.0,
        longitude_max=274.0,
    ),
    event_type="tropical_cyclone",
)
cases = [demo_case]

In [ ]:
forecast = inputs.get_cira_icechunk(
    model_name="FOUR_v200_IFS",
    variables=[derived.TropicalCycloneTrackVariables()],
)

ibtracs_target = ewb.IBTrACS()

In [ ]:
tc_metrics = [
    metrics.LandfallDisplacement(
        approach="first",
        forecast_variable="surface_wind_speed",
        target_variable="surface_wind_speed",
    ),
    metrics.LandfallTimeMeanError(
        approach="first",
        forecast_variable="surface_wind_speed",
        target_variable="surface_wind_speed",
    ),
    metrics.LandfallIntensityMeanAbsoluteError(
        approach="first",
        forecast_variable="surface_wind_speed",
        target_variable="surface_wind_speed",
    ),
]

eval_objects = [
    ewb.EvaluationObject(
        event_type="tropical_cyclone",
        metric_list=tc_metrics,
        target=ibtracs_target,
        forecast=forecast,
    ),
]

runner = ewb.evaluation(
    case_metadata=cases,
    evaluation_objects=eval_objects,
)
outputs = runner.run_evaluation()
print(
    outputs[
        ["metric", "value", "init_time", "case_id_number"]
    ].head(20)
)